In [ ]:
import Pkg; Pkg.add(["Ipopt", "SpecialFunctions"])


In [ ]:
using Random, Distributions, JuMP, Ipopt, SpecialFunctions


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor $L$, value function $V$, migration shares $\mu$, mobility costs $\tau$) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production ($A$, $w$, $\kappa$, $\theta$, $\eta$, $\gamma$, $\alpha$) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$.

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ — discount factor
- $\theta^j$ — Fréchet trade elasticity in sector $j$
- $\nu$ — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$, the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [ ]:
#try indexing format of J[region]_[sector]_[time], x if not indexed by that component

N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1, i.e. sector 0) plus
          # the J real sectors (market columns 2:M, i.e. sectors 1:J). Per the model, a "market"
          # is a region-sector pair (n,j) with j = 0,...,J, so any array indexed over "every
          # market a household could be in" (L, V, mu, tau_mig) has rows = regions (N), columns
          # = markets (M), with column 1 = non-employment. Arrays that only pertain to real
          # production (A, w, kappa, theta, eta, gamma, alpha) keep rows = regions, columns =
          # real sectors (J) -- to look one of these up against a market-indexed array, offset
          # the sector index by +1 (real sector j lives in market column j+1).
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) #labor force in economy at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # Seed for the random number generator (to guarantee reproducibility; this is standard in research these days)

A_0 = rand(N,J) #region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) #arbitrary coefficient

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector (kappa: trade cost)

# sigma = 2 # Substitution elasticity between goods
# L = ones(N, 1) # Size of labor force in each country

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k), j,k = 0,...,J (market columns 1:M,
# where column 1 is non-employment); 0 to stay in the same market, 1 otherwise (tau: migration cost)
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


# Temporary Equilibrium

This section actually **solves** for market-clearing wages (Definition 1: Temporary Equilibrium): given labor supply $L_t$ and fundamentals $(A_t, \kappa_t)$, find wages $w_t = \{w_t^{nj}\}$ solving goods-market clearing and labor-market clearing simultaneously — $N \times J$ equations in $N \times J$ wage unknowns:
$$X_t^{nj} = \alpha^j \sum_{k=1}^J w_t^{nk} L_t^{nk} \qquad\text{and}\qquad w_t^{nj} L_t^{nj} = \sum_{i=1}^N \pi_t^{ij,nj} X_t^{ij}$$

This block is *static* in the sense that matters: it references no other time period, no expectation, no $\beta$ — even though its solution changes every period because $L_t$ does.

**Solved via JuMP + Ipopt**, posed as a feasibility problem: minimize a trivial constant objective subject to the market-clearing equations as nonlinear equality constraints (Ipopt just has to find a feasible point). The system is homogeneous of degree $1$ in $w$ — scaling every wage by $\lambda$ scales both sides of labor-market clearing by $\lambda$, since trade shares $\pi$ depend only on *relative* wages — so wages are only pinned down up to a numeraire. We normalize $w^{11}=1$ and drop that market's own clearing equation, which is redundant with the rest by Walras' law.

In [ ]:
# Temporary equilibrium (Definition 1, PDF Section 5): given labor supply L and the fundamentals
# (A_0, kappa_0), find wages w = {w^{nj}} that clear goods and labor markets simultaneously --
# eq 6 (goods clearing / expenditure) and eq 7 (labor clearing) together, N x J equations in N x J
# wage unknowns. This actually SOLVES for w (via JuMP + Ipopt, posed as a feasibility problem:
# minimize a trivial 0 objective subject to the market-clearing equations as constraints).
#
# The system is homogeneous of degree 1 in w: scaling every wage by a constant lambda scales the
# wage bill (LHS of eq 7) by lambda, and scales expenditure X (eq 6) by lambda too, while trade
# shares pi (eq 5) are unaffected (they only depend on RELATIVE wages across sources). So only
# relative wages are pinned down -- we normalize w[1,1] = 1 and drop that market's own clearing
# equation, which is redundant with the rest by Walras' law (aggregate labor income always equals
# aggregate expenditure here, since there is no trade deficit).

function trade_shares_and_expenditure(w::AbstractMatrix, L::AbstractMatrix)
    x = B .* w # unit cost (eq 4.2)
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    pi = [trade_cost_term[n,j,i] / sum(trade_cost_term[n,j,:]) for n in 1:N, j in 1:J, i in 1:N] # eq 5
    I = vec(sum(w .* L[:, 2:M], dims=2)) # labor income by region (employed markets only, eq analog of I_0 above)
    X = I * alpha' # eq 6
    return pi, X
end

function solve_temporary_equilibrium(L::AbstractMatrix; w_guess = ones(N,J))
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    @variable(model, w[n=1:N, j=1:J] >= 1e-6, start = w_guess[n,j])

    @expression(model, tct[n=1:N, j=1:J, i=1:N],
        (B[i,j]*w[i,j]*kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j]))
    @expression(model, pishare[n=1:N, j=1:J, i=1:N], tct[n,j,i] / sum(tct[n,j,m] for m in 1:N))
    @expression(model, Inc[n=1:N], sum(w[n,k]*L[n,k+1] for k in 1:J))
    @expression(model, X[n=1:N, j=1:J], alpha[j]*Inc[n])

    @constraint(model, w[1,1] == 1.0) # numeraire, replaces the redundant (1,1) equation
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w[n,j]*L[n,j+1] == sum(pishare[i,j,n]*X[i,j] for i in 1:N)) # eq 7
        end
    end

    @objective(model, Min, 0) # feasibility problem: no objective, just satisfy the constraints
    optimize!(model)

    w_star = value.(w)
    pi_star, X_star = trade_shares_and_expenditure(w_star, L)
    return w_star, pi_star, X_star, termination_status(model)
end


In [ ]:
# Solve the temporary equilibrium at the initial labor distribution L_0.

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)

println("Solver status: ", status_temp)
println("Equilibrium wages w*:")
display(w_temp)


# Sequential Equilibrium

This section solves the full dynamic model (Definition 2: Sequential Competitive Equilibrium): given $L_0$ and a (here, constant) path of fundamentals $\{A_t,\kappa_t\}_{t=0}^\infty$, find a sequence $\{L_t, \mu_t, V_t, w_t\}_{t=0}^\infty$ solving **both** the household dynamic block (§3.2–3.4) **and** the temporary equilibrium (§4) at *every* $t$ — unlike Notebook 1's Migration Decision / Labor Dynamics sections, which solved the household block once under a wage assumed to be stationary and never revisited it as $L_t$ evolved.

**Solution method** (an outer relaxation loop over a guessed wage *path* $w_0,\dots,w_T$):
1. Compute flow utility $U_t$ at every $t$ from $w_t$.
2. Solve $V_t$ by **backward induction**: $V_T$ is the fixed point of the stationary Bellman equation (approximating "the economy repeats period $T$'s conditions forever after" as a stand-in for the true infinite continuation beyond the horizon); then for $t=T-1,\dots,0$,
$$V_t^{nj} = \log\left(U_t^{nj}\right) + \nu\log\left(\sum_{i,k}\exp\left(\frac{\beta V_{t+1}^{ik}-\tau^{nj,ik}}{\nu}\right)\right)$$
is a single direct evaluation using the already-known $V_{t+1}$ — no inner fixed-point iteration needed except at the terminal $V_T$.
3. Get migration shares $\mu_t$ from $V_{t+1}$ and simulate $L_t$ forward from $L_0$ via the law of motion.
4. Re-solve the temporary equilibrium at each period's newly-updated $L_t$ to get an updated wage path.

Repeat (with a damped update for stability) until the wage path stops changing.

In [ ]:
# Sequential competitive equilibrium (Definition 2, PDF Section 5): given L_0 and fundamentals
# {A_t, kappa_t} (held constant here -- we're not running a counterfactual shock, just solving
# the baseline path), find {L_t, mu_t, V_t, w_t} for t=0,...,T solving BOTH the household dynamic
# block (Section 3) AND the temporary equilibrium (Section 4) at EVERY t. This differs from the
# "Migration decision" / "Labor dynamics" sections above, which solved the household block once
# under a single wage guessed to be stationary and never revisited it as L evolved.
#
# Solution method (the standard approach for these models, e.g. Caliendo-Dvorkin-Parro): an outer
# relaxation loop over a guessed wage PATH w_0,...,w_T. Given a wage path:
#   1) compute flow utility U_mkt_t at every t from w_t
#   2) solve V_t by BACKWARD induction: V_T is the fixed point of the stationary Bellman equation
#      (an approximation for "the economy repeats period T's conditions forever after", standing
#      in for the true infinite continuation beyond the horizon we can compute); then for
#      t = T-1,...,0, V_t is a single direct evaluation using the already-known V_{t+1} (no inner
#      fixed-point iteration needed once V_{t+1} is known -- only the terminal V_T requires one)
#   3) get migration shares mu_t from V_{t+1} (eq 3) and simulate L_t forward from L_0 (eq 4)
#   4) re-solve the temporary equilibrium at each period's newly-updated L_t to get an updated
#      wage path
# Repeat until the wage path stops changing (damped update for stability).

function flow_utility_mkt(w::AbstractMatrix)
    x = B .* w
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    Gamma = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]
    P = [Gamma[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j]) for n in 1:N, j in 1:J] # eq 4.3
    P_hat = [prod((P[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N]
    C = [w[n,j] / P_hat[n] for n in 1:N, j in 1:J] # real consumption of an employed household
    return hcat(b, C) # N x M, column 1 = non-employment (fixed consumption b^n)
end

function stationary_V(U_mkt::AbstractMatrix; tol=1e-12, maxiter=10_000)
    V = log.(U_mkt)
    for _ in 1:maxiter
        V_next = [
            log(U_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
            for n in 1:N, j in 1:M
        ]
        if maximum(abs.(V_next .- V)) < tol
            V = V_next
            break
        end
        V = V_next
    end
    return V
end

function migration_shares(V_next::AbstractMatrix) # eq 3, given next period's value function
    [
        exp((beta*V_next[i,k] - tau_mig[n,j,i,k]) / nu) /
        sum(exp((beta*V_next[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end

function solve_sequential_equilibrium(L0::AbstractMatrix, Tbar::Int;
                                       max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5)
    W = [ones(N,J) for _ in 0:Tbar] # initial wage-path guess
    L = Vector{Matrix{Float64}}(undef, Tbar+1)
    mu = Vector{Array{Float64,4}}(undef, Tbar)
    V = Vector{Matrix{Float64}}(undef, Tbar+1)

    for outer in 1:max_outer
        # (1)-(2): flow utility and backward induction on V given the current wage-path guess
        U_mkt = [flow_utility_mkt(W[t+1]) for t in 0:Tbar]
        V[Tbar+1] = stationary_V(U_mkt[Tbar+1]) # terminal condition
        for t in Tbar:-1:1
            V[t] = [
                log(U_mkt[t][n,j]) + nu * log(sum(exp((beta*V[t+1][i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
                for n in 1:N, j in 1:M
            ]
        end

        # (3): migration shares and forward labor simulation
        for t in 1:Tbar
            mu[t] = migration_shares(V[t+1])
        end
        L[1] = L0
        for t in 1:Tbar
            L[t+1] = [
                sum(mu[t][i,k,n,j] * L[t][i,k] for i in 1:N, k in 1:M)
                for n in 1:N, j in 1:M
            ]
        end

        # (4): re-solve the temporary equilibrium at each period's updated labor distribution
        W_new = similar(W)
        for t in 0:Tbar
            w_star, _, _, _ = solve_temporary_equilibrium(L[t+1]; w_guess = W[t+1])
            W_new[t+1] = w_star
        end

        diff = maximum(maximum(abs.(W_new[t] .- W[t])) for t in 1:Tbar+1)
        W = [damp .* W_new[t] .+ (1-damp) .* W[t] for t in 1:Tbar+1]
        if diff < tol
            println("Sequential equilibrium converged after $outer outer iterations (max wage change = $diff)")
            break
        end
        if outer == max_outer
            println("Reached max_outer=$max_outer without converging (max wage change = $diff)")
        end
    end

    return (; W, L, mu, V)
end


In [ ]:
# Solve the sequential equilibrium over T periods starting from L_0, and inspect the resulting
# wage and labor paths for region 1.

seq_eq = solve_sequential_equilibrium(L_0, T)

println("\nWage path, region 1 (columns: sector 1, sector 2, sector 3):")
for t in 0:T
    println("t=$t: ", seq_eq.W[t+1][1,:])
end

println("\nLabor path, region 1 (columns: non-employment, sector 1, sector 2, sector 3):")
for t in 0:T
    println("t=$t: ", seq_eq.L[t+1][1,:])
end


## Experiment: a 2-period sequential equilibrium

A minimal case of the algorithm above: `Tbar=1` means periods $t=0,1$ only, with the terminal condition $V_1$ = the stationary fixed point (i.e. "period 1's conditions hold forever after"). This isolates one full pass of the loop (compute $V_1 \to V_0$ by backward induction, get $\mu_0$, simulate $L_1$ from $L_0$, then re-solve wages at both $L_0$ and $L_1$) so it's easy to inspect every intermediate object directly.

In [ ]:
function print_labeled(title, description, M::AbstractMatrix, colnames)
    println(title)
    println("  ", description)
    println("             " * join(rpad.(colnames, 16)))
    for n in 1:size(M,1)
        println("  Region $n:  " * join([rpad(string(round(M[n,c], digits=4)), 16) for c in 1:size(M,2)]))
    end
    println()
end

Tbar2 = 1 # periods t = 0, 1
seq_eq_2 = solve_sequential_equilibrium(L_0, Tbar2)

for t in 0:Tbar2
    println("="^70)
    println("PERIOD t = $t")
    println("="^70)
    print_labeled(
        "Labor distribution L[$t]",
        "mass of households currently in each region-market",
        seq_eq_2.L[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Equilibrium wages w[$t]",
        "market-clearing wage solving goods + labor market clearing",
        seq_eq_2.W[t+1], ["Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Value function V[$t]",
        "expected discounted lifetime utility of a household in each region-market",
        seq_eq_2.V[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
end
